In [ ]:
#1. Cargar el CSV y detectar nulos en precio

import pandas as pd
df = pd.read_csv(r"documento")

# Ver cuántos nulos hay en la columna precio
df["precio"].isnull().sum()

#También podemos ver dónde están:
df[df["precio"].isnull()].head()

In [ ]:
#2. Ver el contexto de esos nulos
#La clave no es solo saber que hay nulos, sino en qué año/mes:

df_nulos = df[df["precio"].isnull()][["año", "mes_numero", "mes", "precio"]]
print(df_nulos)
#Ahí ves si los huecos son aislados o bloques enteros.


In [ ]:
#3. Decidir la estrategia (esto es lo importante)

#Opciones típicas para una serie histórica de tarifas:

# >Eliminar filas con precio nulo

# >Rellenar con el último precio conocido (forward fill)

# >Rellenar con el siguiente precio conocido (backward fill)

# >Interpolar (si tuviera sentido)

# >Marcar los nulos y no rellenar (para análisis cualitativo)



In [ ]:
#4. Opción A — Eliminar filas con precio nulo
#Útil si son pocos casos y no rompen la serie.
df_drop = df.dropna(subset=["precio"])

In [ ]:
#5. Opción B — Rellenar con el último precio conocido (forward fill)
#Tiene mucho sentido en tarifas:
#si no hay dato, asumimos que se mantiene el precio anterior.

df_sorted = df.sort_values(["año", "mes_numero"])
df_ffill = df_sorted.copy()
df_ffill["precio"] = df_ffill["precio"].ffill()

#Si al principio también hay nulos, podés luego hacer un bfill:

df_ffill["precio"] = df_ffill["precio"].bfill()

In [ ]:
#6. Opción C — Rellenar con el siguiente precio conocido (backward fill)
#Menos habitual, pero posible:

df_bfill = df.sort_values(["año", "mes_numero"]).copy()
df_bfill["precio"] = df_bfill["precio"].bfill()

In [ ]:
#7. Opción D — Interpolación numérica
#Si quisieras “suavizar” entre dos precios conocidos (no muy realista para tarifas, pero útil en otras series):

df_interp = df.sort_values(["año", "mes_numero"]).copy()
df_interp["precio"] = df_interp["precio"].interpolate(method="linear")

In [ ]:
#8. Opción E — Mantener los nulos pero marcarlos
#A veces no querés inventar datos, solo saber dónde falta información:

df["precio_es_nulo"] = df["precio"].isnull()
#Y luego:

df["precio_ffill"] = df["precio"].ffill()
#Así tenés la serie original y una serie “rellena” para gráficos.

In [ ]:
#9. Verificar después de la limpieza
#Siempre, después de aplicar una estrategia:

df_limpio = df_ffill  # o la versión que elijas
df_limpio["precio"].isnull().sum()
#Debe dar 0 si decidiste no dejar nulos.